In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import auc, classification_report, confusion_matrix, roc_curve
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")


In [7]:
train = pd.read_csv('archive/train.csv')
test = pd.read_csv('archive/test.csv')


In [9]:
grade_features = ['GPA', 'TestScore_Math', 'TestScore_Reading', 'TestScore_Science']

# 범주형 변수 인코딩
le_race = LabelEncoder()
le_gender = LabelEncoder()

y_race_train = le_race.fit_transform(train['Race'])
y_race_test = le_race.fit_transform(test['Race'])
x_train = train[grade_features]

y_gender_test = le_race.fit_transform(test['Gender'])
y_gender_train = le_gender.fit_transform(train['Gender'])   
x_test = test[grade_features]





In [10]:
race_model = xgb.XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42
)

# 성별 예측 모델
gender_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

# 하이퍼파라미터 튜닝
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 200, 300],
    'subsample': [0.8, 0.9, 1.0]
}

In [ ]:
print("인종 예측 모델 하이퍼파라미터 튜닝 중...")
# 인종 예측 모델 튜닝
race_grid = GridSearchCV(race_model, param_grid, cv=5, scoring='f1_weighted')
race_grid.fit(x_train, y_race_train)

print("성별 예측 모델 하이퍼파라미터 튜닝 중...")
# 성별 예측 모델 튜닝
gender_grid = GridSearchCV(gender_model, param_grid, cv=5, scoring='f1')
gender_grid.fit(x_train, y_gender_train)

# 최적 모델 선택
best_race_model = race_grid.best_estimator_
best_gender_model = gender_grid.best_estimator_

# 3. 모델 평가
# 인종 예측 평가
race_pred = best_race_model.predict(x_test)
race_pred_proba = best_race_model.predict_proba(x_test)

print("\n인종 예측 모델 성능:")
print(classification_report(y_race_test, race_pred, target_names=le_race.classes_))

# 성별 예측 평가
gender_pred = best_gender_model.predict(x_test)
gender_pred_proba = best_gender_model.predict_proba(x_test)

print("\n성별 예측 모델 성능:")
print(classification_report(y_gender_test, gender_pred, target_names=le_gender.classes_))

# 4. 시각화
# 특성 중요도 플롯
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
xgb.plot_importance(best_race_model, title='인종 예측 특성 중요도')
plt.subplot(1, 2, 2)
xgb.plot_importance(best_gender_model, title='성별 예측 특성 중요도')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.close()

# 혼동 행렬
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.heatmap(confusion_matrix(y_race_test, race_pred), 
            annot=True, fmt='d', cmap='Blues',
            xticklabels=le_race.classes_,
            yticklabels=le_race.classes_)
plt.title('인종 예측 혼동 행렬')
plt.xlabel('예측')
plt.ylabel('실제')

plt.subplot(1, 2, 2)
sns.heatmap(confusion_matrix(y_gender_test, gender_pred), 
            annot=True, fmt='d', cmap='Blues',
            xticklabels=le_gender.classes_,
            yticklabels=le_gender.classes_)
plt.title('성별 예측 혼동 행렬')
plt.xlabel('예측')
plt.ylabel('실제')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.close()

# ROC 커브 (성별 예측의 경우)
fpr, tpr, _ = roc_curve(y_gender_test, gender_pred_proba[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('성별 예측 ROC 커브')
plt.legend(loc="lower right")
plt.savefig('roc_curve.png')
plt.close()

# 결과 저장
results = {
    'race_model': {
        'best_params': race_grid.best_params_,
        'best_score': race_grid.best_score_,
        'feature_importance': dict(zip(grade_features, best_race_model.feature_importances_))
    },
    'gender_model': {
        'best_params': gender_grid.best_params_,
        'best_score': gender_grid.best_score_,
        'feature_importance': dict(zip(grade_features, best_gender_model.feature_importances_))
    }
}

# 결과를 DataFrame으로 변환하여 저장
race_importance = pd.DataFrame({
    'Feature': grade_features,
    'Importance': best_race_model.feature_importances_
}).sort_values('Importance', ascending=False)

gender_importance = pd.DataFrame({
    'Feature': grade_features,
    'Importance': best_gender_model.feature_importances_
}).sort_values('Importance', ascending=False)

race_importance.to_csv('race_prediction_importance.csv', index=False)
gender_importance.to_csv('gender_prediction_importance.csv', index=False)

# 최종 결과 출력
print("\n=== 최종 분석 결과 ===")
print("\n1. 인종 예측 모델")
print(f"최적 하이퍼파라미터: {race_grid.best_params_}")
print(f"최적 교차 검증 점수: {race_grid.best_score_:.3f}")
print("\n특성 중요도:")
print(race_importance)

print("\n2. 성별 예측 모델")
print(f"최적 하이퍼파라미터: {gender_grid.best_params_}")
print(f"최적 교차 검증 점수: {gender_grid.best_score_:.3f}")
print("\n특성 중요도:")
print(gender_importance) 

인종 예측 모델 하이퍼파라미터 튜닝 중...


c:\anaconda\anaconda3\envs\math\lib\site-packages\xgboost\training.py:183: UserWarning: [13:17:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\anaconda\anaconda3\envs\math\lib\site-packages\xgboost\training.py:183: UserWarning: [13:18:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\anaconda\anaconda3\envs\math\lib\site-packages\xgboost\training.py:183: UserWarning: [13:20:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\anaconda\anaconda3\envs\math\lib\site-packages\xgboost\training.py:183: UserWarning: [13:21:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are